## TODOs
- fix sentiment, commit, specific
- eval results
- calc score
- new class for analyze results?
- improve chunks/preprocess 
    - bug: at end of chunk repeat words words words words words words
    - sentence splitting sometimes gets mangled

## OPT
- eval filter makes sense? add confidence threshhold (eg >.8)
    - analyzer.inspect_filter(pdf_path, n_samples=2)
- eval translation quality
- simplify api:    def filter_climate_chunks(self, data: Union[Dict, str], batch_size=None) -> Dict:
    - only use on data-type not union.

## analyzer class

before rerunning if GPU oom crash.

In [ ]:
# before rerunning after GPU out-of-memory (oom) crash.
import subprocess
subprocess.run(["nvidia-smi"], capture_output=True) # check if sleep
# subprocess.run(["nvidia-smi", "--gpu-reset"], capture_output=True) # reset if not detected
# subprocess.run(["nvidia-smi"], "sudo nvidia-smi -pm 1", capture_output=True) # wake up if sleepy/off

!echo $LD_LIBRARY_PATH # test cuda in path?
!echo $PATH | grep cuda | cut -d: -f9

- run befor script. to test if GPU supported!

In [6]:
import os
# === CRITICAL: Must be set BEFORE importing torch ===
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6,max_split_size_mb:128"

import torch


if torch.cuda.is_available():
    # print(f"CUDA available: {torch.cuda.is_available()}")

    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("✅ SUCCESS!")

    print(f"ALLOC_CONF: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF')}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    # Test allocation
    torch.cuda.empty_cache()
    print(f"Free VRAM before: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")
else:
    print("❌ NO CUDA / GPU found!")


PyTorch version: 2.9.1+cu130
CUDA version: 13.0
GPU: NVIDIA T1200 Laptop GPU
✅ SUCCESS!
ALLOC_CONF: expandable_segments:True,garbage_collection_threshold:0.6,max_split_size_mb:128
Total VRAM: 3.63 GB
Free VRAM before: 1.80 GB


In [ ]:
"""
ClimateReportAnalyzer - Integrated Simplified Version

CHANGES FROM ORIGINAL:
1. Simplified _prep_text() - respects sentence boundaries
2. Simplified _chunk_text() - cleaner logic
3. Consolidated _is_noise_line() - all noise detection in one place
4. Added _clean_artifacts() - replaces _clean_source_text()
5. Simplified _detect_severe_repetition() - faster check
6. Added post-translation cleaning in _translate_chunks_helsinki()

KEPT FROM ORIGINAL:
- All __init__ and infrastructure
- All BERT analysis methods
- All caching methods
- process_pdfs() batch processing
- Metadata extraction
- GPU memory management
"""

import spacy
import langid
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer, pipeline
import fitz  # PyMuPDF
import torch
import json
import re
import unicodedata
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
import time
from collections import Counter
import gc
import os

# === CRITICAL: Must be set BEFORE importing torch ===
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.7,max_split_size_mb:256"

_nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer", "tagger"])
_nlp.max_length = 2_000_000


class ClimateReportAnalyzer:
    # Chunking settings
    MIN_CHARS = 600
    MAX_CHARS = 1600
    MAX_TOKENS = MAX_CHARS // 4
    BATCH_SIZE = 32   # For BERT models

    # Translation settings - OPTIMIZED FOR HELSINKI + 4GB VRAM
    TRANSLATE_BATCH_SIZE = 32
    TRANSLATE_INPUT_MAX_TOKENS = 512
    TRANSLATE_OUTPUT_MAX_TOKENS = 512

    # Helsinki-NLP models (fast, language-specific)
    HELSINKI_MODELS = {
        'de': 'Helsinki-NLP/opus-mt-de-en',
        'es': 'Helsinki-NLP/opus-mt-es-en',
        'it': 'Helsinki-NLP/opus-mt-it-en',
        'zh': 'Helsinki-NLP/opus-mt-zh-en',
        'fr': 'Helsinki-NLP/opus-mt-fr-en',
        'nl': 'Helsinki-NLP/opus-mt-nl-en',
        'pt': 'Helsinki-NLP/opus-mt-ROMANCE-en',
        'pl': 'Helsinki-NLP/opus-mt-pl-en',
        'ru': 'Helsinki-NLP/opus-mt-ru-en',
        'ja': 'Helsinki-NLP/opus-mt-ja-en',
        'ko': 'Helsinki-NLP/opus-mt-ko-en',
    }

    # Characters considered as bullet/decoration spam
    SPAM_CHARS = set('•·*─―–-')

    def __init__(self, cache_dir="cache", use_torch_compile=True):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
        self.pdf_path = None

        self.use_torch_compile = use_torch_compile

        self.device = 0 if torch.cuda.is_available() else -1
        self.models = {}
        self.translator = None
        self.translator_lang = None

        # In-memory cache
        self.prep_data = None
        self.bert_data = None

        # Report GPU status
        if self.device >= 0:
            gpu_name = torch.cuda.get_device_name(0)
            total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"✓ GPU: {gpu_name} ({total_mem:.1f}GB)")
            print(
                f"✓ PYTORCH_CUDA_ALLOC_CONF: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'NOT SET')}")
            print(f"✓ Translation backend: 'Helsinki-NLP (fast)'")
            self._clear_gpu_memory()

    def _get_gpu_memory_info(self) -> Dict:
        """Get current GPU memory status."""
        if self.device < 0:
            return {'available': False}
        return {
            'available': True,
            'allocated_gb': torch.cuda.memory_allocated(0) / 1e9,
            'reserved_gb': torch.cuda.memory_reserved(0) / 1e9,
            'total_gb': torch.cuda.get_device_properties(0).total_memory / 1e9,
            'free_gb': (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
        }

    def _clear_gpu_memory(self):
        """Standard GPU memory cleanup."""
        if self.device >= 0:
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    def _emergency_gpu_cleanup(self):
        """Aggressive GPU cleanup for OOM recovery."""
        if self.device < 0:
            return

        before = self._get_gpu_memory_info()
        print(
            f"  Emergency cleanup: {before['allocated_gb']:.2f}GB allocated before")

        # Unload all models
        if self.translator:
            try:
                if 'model' in self.translator:
                    self.translator['model'].cpu()
                    del self.translator['model']
                if 'tokenizer' in self.translator:
                    del self.translator['tokenizer']
            except:
                pass
            self.translator = None
            self.translator_lang = None

        for name in list(self.models.keys()):
            try:
                self.models[name].model.cpu()
                del self.models[name]
            except:
                pass
        self.models = {}

        gc.collect()

        # Move any remaining CUDA tensors to CPU
        for obj in gc.get_objects():
            try:
                if torch.is_tensor(obj) and obj.is_cuda:
                    obj.data = obj.data.cpu()
            except:
                pass

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        gc.collect()

        after = self._get_gpu_memory_info()
        print(
            f"  Emergency cleanup: {after['allocated_gb']:.2f}GB allocated after")

    def _unload_translator(self):
        """Unload translation model to free memory."""
        if self.translator:
            print("  Unloading translation model...")
            try:
                if 'model' in self.translator:
                    self.translator['model'].cpu()
                    del self.translator['model']
                if 'tokenizer' in self.translator:
                    del self.translator['tokenizer']
            except Exception as e:
                print(f"  ⚠ Error during unload: {e}")

            self.translator = None
            self.translator_lang = None
            self._clear_gpu_memory()

            info = self._get_gpu_memory_info()
            print(f"  GPU memory after unload: {info['allocated_gb']:.2f}GB")

    # ==================== HELSINKI-NLP TRANSLATION ====================

    def _load_helsinki_translator(self, src_lang: str):
        """Load Helsinki-NLP model for specific language."""
        # Check if correct model already loaded
        if self.translator and self.translator_lang == src_lang:
            return self.translator

        # Unload previous model if different language
        if self.translator:
            self._unload_translator()

        model_name = self.HELSINKI_MODELS.get(src_lang)
        if not model_name:
            return None

        print(f"Loading Helsinki-NLP model: {model_name}")
        start = time.time()

        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device >= 0 else torch.float32,
        )

        if self.device >= 0:
            model = model.cuda()

            # Optional: torch.compile for extra speed (PyTorch 2.0+)
            if self.use_torch_compile and hasattr(torch, 'compile'):
                try:
                    print("  Applying torch.compile()...")
                    model = torch.compile(model, mode="reduce-overhead")
                except Exception as e:
                    print(f"  torch.compile failed: {e}")

        model.eval()

        load_time = time.time() - start
        info = self._get_gpu_memory_info()
        print(
            f"✓ Helsinki model loaded in {load_time:.1f}s ({info['allocated_gb']:.2f}GB)")

        self.translator = {'model': model,
                           'tokenizer': tokenizer, 'type': 'helsinki'}
        self.translator_lang = src_lang
        return self.translator

    def _translate_chunks_helsinki(self, chunks: List[str], src_lang: str) -> List[str]:
        """Translate using Helsinki-NLP with repetition mitigation + post-cleaning."""
        translator = self._load_helsinki_translator(src_lang)
        model = translator['model']
        tokenizer = translator['tokenizer']

        print(f"Translating {src_lang} → en (Helsinki-NLP)")
        print(f"  Repetition mitigation: no_repeat_ngram_size=3, repetition_penalty=1.2, num_beams=4")

        translated = []
        batch_size = max(8, self.TRANSLATE_BATCH_SIZE //
                         2) if self.TRANSLATE_BATCH_SIZE > 16 else self.TRANSLATE_BATCH_SIZE

        with torch.no_grad():
            for batch_idx, i in enumerate(tqdm(range(0, len(chunks), batch_size), desc="Translating")):
                batch = chunks[i:i + batch_size]

                try:
                    inputs = tokenizer(
                        batch,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=self.TRANSLATE_INPUT_MAX_TOKENS
                    )

                    if self.device >= 0:
                        inputs = {k: v.cuda() for k, v in inputs.items()}

                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=self.TRANSLATE_OUTPUT_MAX_TOKENS,
                        no_repeat_ngram_size=3,
                        repetition_penalty=1.2,
                        num_beams=4,
                        early_stopping=True,
                        do_sample=False,
                        use_cache=True,
                    )

                    batch_translations = tokenizer.batch_decode(
                        outputs, skip_special_tokens=True)
                    translated.extend(batch_translations)

                    del inputs, outputs

                except RuntimeError as e:
                    if "out of memory" in str(e):
                        print(
                            f"\n⚠ OOM at batch {batch_idx}, falling back to greedy...")
                        torch.cuda.empty_cache()

                        for chunk in batch:
                            try:
                                inputs = tokenizer([chunk], return_tensors="pt",
                                                   truncation=True, max_length=self.TRANSLATE_INPUT_MAX_TOKENS)
                                if self.device >= 0:
                                    inputs = {k: v.cuda()
                                              for k, v in inputs.items()}
                                output = model.generate(
                                    **inputs,
                                    max_new_tokens=self.TRANSLATE_OUTPUT_MAX_TOKENS,
                                    no_repeat_ngram_size=3,
                                    repetition_penalty=1.2,
                                    num_beams=1,
                                    do_sample=False
                                )
                                translated.append(tokenizer.decode(
                                    output[0], skip_special_tokens=True))
                                del inputs, output
                            except:
                                translated.append(chunk)
                                torch.cuda.empty_cache()
                    else:
                        raise

        # ========== NEW: POST-TRANSLATION CLEANING ==========
        print("Post-processing translations...")
        cleaned = []
        skipped = 0

        for trans in translated:
            # Clean artifacts
            trans = self._clean_artifacts(trans)

            # Check for severe repetition
            if self._detect_severe_repetition(trans):
                # Try to salvage
                trans = self._remove_repetitions(trans, max_repeat=2)

                # If still bad after cleaning, skip
                if self._detect_severe_repetition(trans):
                    skipped += 1
                    continue

            cleaned.append(trans)

        if skipped > 0:
            print(
                f"  ⚠ Skipped {skipped} chunks with severe repetition ({skipped/len(translated)*100:.1f}%)")

        return cleaned

    # ==================== SIMPLIFIED TEXT PROCESSING ====================

    def _fix_pdf_encoding(self, text: str) -> str:
        """Fix common PDF encoding issues."""
        text = unicodedata.normalize('NFC', text)
        text = re.sub(r'\(cid:\d+\)', '', text)
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)
        for char in ['\u00ad', '\u200b', '\u200c', '\u200d', '\ufeff']:
            text = text.replace(char, '')
        return text

    def _is_noise_line(self, line: str) -> bool:
        """SIMPLIFIED: Single function to detect all types of noise."""
        line = line.strip()

        # Too short
        if len(line) < 3:
            return True

        # Spaced-out headers (G E S C H Ä F T S B E R I C H T)
        # Check if mostly single letters/chars with spaces
        words = line.split()
        if len(words) > 5:
            single_chars = sum(1 for w in words if len(w) <= 2)
            if single_chars / len(words) > 0.7:  # 70%+ are 1-2 chars
                return True

        # TOC entries (dots with page numbers)
        if re.match(r'^.{5,50}\.{5,}\s*\d+$', line):
            return True

        # Page numbers
        if re.match(r'^(page|p\.?)\s*\d+|^\d+\s*(of|/)\s*\d+$', line, re.I):
            return True

        # Pure numbers/dates (short)
        if re.match(r'^[\d\s\.\-\/]+$', line) and len(line) < 15:
            return True

        # Number-heavy sequences (likely table data)
        # If >60% of words are numbers, it's probably a mangled table
        if len(words) > 8:
            num_count = sum(1 for w in words if re.match(
                r'^\d+[\.\,]?\d*$', w))
            if num_count / len(words) > 0.6:
                return True

        # URLs
        if re.match(r'^https?://\S+$', line, re.I):
            return True

        # Excessive bullets/decoration
        spam_count = sum(1 for c in line if c in self.SPAM_CHARS)
        if spam_count > 5:
            return True

        # High dot ratio (TOC artifacts)
        if len(line) > 10 and line.count('.') / len(line) > 0.4:
            return True

        # Country/entity lists (likely mangled tables)
        if re.match(r'^[A-Z][a-z]+(\s+[A-Z][a-z]+){10,}$', line):
            return True

        return False

    def _clean_artifacts(self, text: str) -> str:
        """SIMPLIFIED: Clean common PDF artifacts."""
        # Fix spaced-out headers (G E S C H Ä F T → GESCHÄFT)
        # Look for sequences of single letters/short words with spaces
        text = re.sub(
            r'\b([A-ZÄÖÜ])\s+([A-ZÄÖÜ])\s+([A-ZÄÖÜ])', r'\1\2\3', text)

        # Remove excessive dot sequences (ToC artifacts)
        text = re.sub(r'\.{5,}', '...', text)

        # Remove repeated punctuation/special chars
        text = re.sub(r'([.\-–—•·])\1{3,}', r'\1\1', text)

        # Collapse excessive whitespace
        text = re.sub(r'\s{3,}', ' ', text)

        return text.strip()

    def _detect_severe_repetition(self, text: str) -> bool:
        """SIMPLIFIED: Fast check for pathological repetition."""
        words = text.split()
        if len(words) < 10:
            return False

        # Check for excessive single-word repetition (e.g., "Business Business Business")
        # Look for any word repeated 5+ times consecutively
        for i in range(len(words) - 4):
            if words[i] == words[i+1] == words[i+2] == words[i+3] == words[i+4]:
                return True

        # Check if any non-stopword appears > 30% of the time
        stopwords = {'the', 'a', 'an', 'and', 'or', 'of', 'to', 'in', 'for',
                     'on', 'with', 'is', 'are', 'was', 'were', 'be', 'been'}

        word_counts = Counter(w.lower()
                              for w in words if w.lower() not in stopwords)

        if not word_counts:
            return False

        most_common_word, most_common_count = word_counts.most_common(1)[0]
        content_words = sum(word_counts.values())

        return content_words > 0 and (most_common_count / content_words) > 0.3

    def _remove_repetitions(self, text: str, max_repeat: int = 2) -> str:
        """Remove excessive consecutive repetitions."""
        # Remove character spam sequences
        for char in self.SPAM_CHARS:
            pattern = rf'(\s*{re.escape(char)}\s*){{3,}}'
            text = re.sub(pattern, f' {char} ', text)

        text = re.sub(r'\s+', ' ', text).strip()

        # Remove consecutive identical words
        words = text.split()
        if len(words) < 4:
            return text

        result = []
        i = 0
        while i < len(words):
            word = words[i]
            count = 1
            while i + count < len(words) and words[i + count] == word:
                count += 1
            result.extend([word] * min(count, max_repeat))
            i += count

        text = ' '.join(result)
        words = text.split()

        # Remove repeated n-gram phrases
        for n in [4, 3, 2]:
            if len(words) < n * 2:
                continue

            cleaned_words = []
            i = 0
            while i < len(words):
                if i + n * 2 <= len(words):
                    ngram = ' '.join(words[i:i+n])
                    next_ngram = ' '.join(words[i+n:i+n*2])

                    if ngram == next_ngram:
                        cleaned_words.extend(words[i:i+n])
                        j = i + n
                        while j + n <= len(words) and ' '.join(words[j:j+n]) == ngram:
                            j += n
                        i = j
                    else:
                        cleaned_words.append(words[i])
                        i += 1
                else:
                    cleaned_words.append(words[i])
                    i += 1

            words = cleaned_words

        return ' '.join(words)

    def _prep_text(self, raw_text: str) -> str:
        """
        SIMPLIFIED: Preprocess with intelligent paragraph reconstruction.
        KEY: Respects sentence boundaries to avoid mid-sentence breaks.
        """
        # Fix encoding
        text = self._fix_pdf_encoding(raw_text)

        # Fix hyphenation (geo-\ngraphy → geography)
        text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)

        # Process line by line
        lines = []
        for line in text.split('\n'):
            line = re.sub(r'[ \t]+', ' ', line).strip()

            if not line:
                lines.append('')
            elif self._is_noise_line(line):
                continue  # Skip noise
            else:
                lines.append(line)

        # Reconstruct paragraphs intelligently
        paragraphs = []
        current = []

        for i, line in enumerate(lines):
            if line:
                current.append(line)

                # Check if this line completes a sentence
                ends_sentence = re.search(r'[.!?]\s*$', line)

                # If next line is blank and current ends sentence, finalize paragraph
                if ends_sentence and (i + 1 >= len(lines) or not lines[i + 1]):
                    para = ' '.join(current)
                    para = self._clean_artifacts(para)

                    # Only keep substantial, non-repetitive paragraphs
                    if len(para) > 30 and not self._detect_severe_repetition(para):
                        paragraphs.append(para)
                    current = []

            elif current:  # Blank line encountered
                # Only break if last line ended with sentence punctuation
                if current and re.search(r'[.!?]\s*$', current[-1]):
                    para = ' '.join(current)
                    para = self._clean_artifacts(para)

                    if len(para) > 30 and not self._detect_severe_repetition(para):
                        paragraphs.append(para)
                    current = []
                # Otherwise keep accumulating (might be mid-sentence column break)

        # Flush remaining
        if current:
            para = ' '.join(current)
            para = self._clean_artifacts(para)

            if len(para) > 30 and not self._detect_severe_repetition(para):
                paragraphs.append(para)

        return '\n\n'.join(paragraphs)

    def _split_sentences(self, text: str) -> List[str]:
        """Split text into sentences using spaCy."""
        global _nlp
        doc = _nlp(text)
        return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    def _chunk_text(self, text: str) -> List[str]:
        """
        SIMPLIFIED: Clear chunking logic.
        1. Perfect size → use or merge
        2. Too large → split by sentences
        3. Too small → accumulate
        """
        paragraphs = re.split(r'\n\s*\n', text)
        paragraphs = [p.strip() for p in paragraphs if p.strip()]

        if not paragraphs:
            return []

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para_len = len(para)

            # Perfect size - use as-is or try to merge
            if self.MIN_CHARS <= para_len <= self.MAX_CHARS:
                if current_chunk:
                    merged = current_chunk + " " + para
                    if len(merged) <= self.MAX_CHARS:
                        current_chunk = merged
                    else:
                        # Can't merge, flush current
                        chunks.append(current_chunk)
                        current_chunk = para
                else:
                    current_chunk = para

            # Too large - must split by sentences
            elif para_len > self.MAX_CHARS:
                # Flush any pending chunk first
                if current_chunk:
                    chunks.append(current_chunk)
                    current_chunk = ""

                # Split this paragraph into sentence-based chunks
                sentences = self._split_sentences(para)
                temp_chunk = ""

                for sent in sentences:
                    if not temp_chunk:
                        temp_chunk = sent
                    elif len(temp_chunk) + len(sent) + 1 <= self.MAX_CHARS:
                        temp_chunk += " " + sent
                    else:
                        # Flush temp_chunk if it's big enough
                        if len(temp_chunk) >= self.MIN_CHARS:
                            chunks.append(temp_chunk)
                        temp_chunk = sent

                # Handle remaining
                if temp_chunk:
                    if len(temp_chunk) >= self.MIN_CHARS:
                        chunks.append(temp_chunk)
                    else:
                        current_chunk = temp_chunk  # Carry forward

            # Too small - accumulate
            else:
                if current_chunk:
                    current_chunk += " " + para
                else:
                    current_chunk = para

        # Flush final chunk if substantial
        if current_chunk and len(current_chunk) >= self.MIN_CHARS:
            chunks.append(current_chunk)

        return chunks

    def _extract_text_pymupdf(self) -> str:
        """Extract text using PyMuPDF with sorted blocks and table detection."""
        all_text = []

        try:
            doc = fitz.open(self.pdf_path)
        except Exception as e:
            print(f"  ⚠ PyMuPDF failed to open: {e}")
            return ""

        for page in tqdm(doc, desc="Extracting (PyMuPDF)"):
            # Detect tables on this page
            table_bboxes = []
            try:
                tables = page.find_tables()
                if tables:
                    table_bboxes = [table.bbox for table in tables]
            except:
                pass  # Table detection not available or failed

            try:
                blocks = page.get_text("dict", sort=True)["blocks"]
            except Exception:
                text = page.get_text("text")
                if text:
                    all_text.append(text)
                continue

            page_paragraphs = []

            for block in blocks:
                if block.get("type") != 0:
                    continue

                bbox = block.get("bbox", [0, 0, 0, 0])
                block_height = bbox[3] - bbox[1]
                if block_height < 5:
                    continue

                # Skip blocks that overlap with detected tables
                is_in_table = False
                for table_bbox in table_bboxes:
                    if self._bbox_overlap(bbox, table_bbox, threshold=0.5): # low = more aggressive, skips more content, hi = more conservative, keeps borderline content
                        is_in_table = True
                        break

                if is_in_table:
                    continue  # Skip table content

                block_lines = []
                for line in block.get("lines", []):
                    spans_text = []
                    for span in line.get("spans", []):
                        span_text = span.get("text", "").strip()
                        if span_text:
                            spans_text.append(span_text)

                    if spans_text:
                        line_text = " ".join(spans_text)
                        block_lines.append(line_text)

                if block_lines:
                    paragraph = " ".join(block_lines)
                    paragraph = re.sub(r'\s+', ' ', paragraph).strip()

                    if paragraph and len(paragraph) > 2:
                        page_paragraphs.append(paragraph)

            if page_paragraphs:
                all_text.append("\n\n".join(page_paragraphs))

        doc.close()
        return "\n\n".join(all_text)

    def _bbox_overlap(self, bbox1, bbox2, threshold=0.5):
        """Check if two bounding boxes overlap significantly."""
        x1_min, y1_min, x1_max, y1_max = bbox1
        x2_min, y2_min, x2_max, y2_max = bbox2

        # Calculate intersection
        x_overlap = max(0, min(x1_max, x2_max) - max(x1_min, x2_min))
        y_overlap = max(0, min(y1_max, y2_max) - max(y1_min, y2_min))

        if x_overlap == 0 or y_overlap == 0:
            return False

        intersection_area = x_overlap * y_overlap
        bbox1_area = (x1_max - x1_min) * (y1_max - y1_min)

        # If bbox1 overlaps >threshold with table, consider it part of table
        return (intersection_area / bbox1_area) > threshold if bbox1_area > 0 else False

    # ==================== METADATA & CACHING (UNCHANGED) ====================

    def _extract_metadata_from_path(self, pdf_path: Path) -> dict:
        """Extract company, year, ID from path."""
        pdf_path = Path(pdf_path)
        filename = pdf_path.stem
        parts = pdf_path.parts

        company = None
        for i, part in enumerate(parts):
            if part.lower() == 'reports' and i + 1 < len(parts):
                company = parts[i + 1]
                break

        if not company:
            skip_folders = {'factsheets', 'highlights',
                            'annual', 'reports', 'data', 'pdfs'}
            for parent in [pdf_path.parent, pdf_path.parent.parent]:
                if parent.name.lower() not in skip_folders and parent.name:
                    company = parent.name
                    break

        company_id = None
        match = re.match(r'^(\d{2,3})_', filename)
        if match:
            company_id = match.group(1)

        year = None
        year_matches = re.findall(r'(20[12]\d)', filename)
        if year_matches:
            year = int(year_matches[0])

        return {'company': company, 'company_id': company_id, 'year': year}

    def set_pdf_path(self, pdf_path: str) -> None:
        """Set current PDF path."""
        new_path = Path(pdf_path)
        if self.pdf_path != new_path:
            self.pdf_path = new_path
            self.prep_data = None
            self.bert_data = None

    def _get_cache_path(self, suffix: str) -> Path:
        """Get cache file path."""
        if not self.pdf_path:
            raise ValueError("No PDF path set")
        return self.cache_dir / f"{self.pdf_path.stem}_{suffix}.json"

    def _load_prep(self) -> Optional[Dict]:
        """Load prep data from cache."""
        if self.prep_data is not None:
            return self.prep_data
        cache_file = self._get_cache_path('prep')
        if cache_file.exists():
            print(f"✓ Loading prep from cache: {cache_file.name}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                self.prep_data = json.load(f)
            return self.prep_data
        return None

    def _save_prep(self) -> None:
        """Save prep data to cache."""
        if self.prep_data is None:
            return
        cache_file = self._get_cache_path('prep')
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(self.prep_data, f, ensure_ascii=False, indent=2)
        print(f"✓ Cached prep: {cache_file.name}")

    def _load_bert(self) -> Optional[Dict]:
        """Load BERT analysis from cache."""
        if self.bert_data is not None:
            return self.bert_data
        cache_file = self._get_cache_path('bert')
        if cache_file.exists():
            print(f"✓ Loading BERT analysis from cache: {cache_file.name}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                self.bert_data = json.load(f)
            return self.bert_data
        return None

    def _save_bert(self) -> None:
        """Save BERT analysis to cache."""
        if self.bert_data is None:
            return
        cache_file = self._get_cache_path('bert')
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(self.bert_data, f, ensure_ascii=False, indent=2)
        print(f"✓ Cached BERT analysis: {cache_file.name}")

    def _detect_language(self, text: str) -> str:
        """Detect language from text."""
        try:
            text_len = len(text)
            samples = []
            if text_len > 15000:
                samples.append(text[1000:4000])
                samples.append(text[text_len//2:text_len//2+3000])
                samples.append(text[-4000:-1000])
            else:
                samples.append(text[100:min(5000, text_len-100)])

            votes = []
            for sample in samples:
                sample_clean = re.sub(r'[^\w\s]', ' ', sample)
                sample_clean = re.sub(r'\s+', ' ', sample_clean)
                if len(sample_clean) > 100:
                    lang, _ = langid.classify(sample_clean)
                    votes.append(lang)

            if votes:
                lang = Counter(votes).most_common(1)[0][0]
                print(f"✓ Language detected: {lang}")
                return lang
            return 'en'
        except Exception as e:
            print(f"⚠ Language detection failed: {e}")
            return 'en'

    # ==================== MAIN PIPELINE (UNCHANGED STRUCTURE) ====================

    def extract_pdf(self) -> Dict:
        """Extract and chunk PDF text."""
        if not self.pdf_path:
            raise ValueError("No PDF path set")
        if self.pdf_path.is_dir():
            raise ValueError("Cannot extract from directory")

        if self.prep_data is not None:
            return self.prep_data

        cached = self._load_prep()
        if cached:
            return cached

        print(f"\n{'='*80}")
        print(f"STEP 1: EXTRACTING PDF")
        print(f"{'='*80}\n")

        metadata = self._extract_metadata_from_path(self.pdf_path)
        print(
            f"✓ Metadata: company={metadata['company']}, id={metadata['company_id']}, year={metadata['year']}")

        raw_text = self._extract_text_pymupdf()
        print(f"✓ Raw extraction: {len(raw_text):,} characters")

        text = self._prep_text(raw_text)
        print(f"✓ After cleaning: {len(text):,} characters")

        lang = self._detect_language(text)
        chunks = self._chunk_text(text)
        print(f"✓ Created {len(chunks)} chunks")

        if chunks:
            lengths = [len(c) for c in chunks]
            print(
                f"  Chunk sizes: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")

        chunk_ids = []
        for idx in range(len(chunks)):
            if metadata['company_id']:
                chunk_ids.append(f"{metadata['company_id']}_{idx:03d}")
            else:
                chunk_ids.append(f"chunk_{idx:03d}")

        self.prep_data = {
            "pdf_path": str(self.pdf_path),
            "company": metadata['company'],
            "company_id": metadata['company_id'],
            "year": metadata['year'],
            "language": lang,
            "extraction_method": "pymupdf_blocks",
            "num_chunks": len(chunks),
            "chunks": chunks,
            "chunk_ids": chunk_ids,
            "translated": False,
            "extracted_at": str(datetime.now())
        }

        self._save_prep()
        return self.prep_data

    def translate_pdf(self) -> Dict:
        """Translate extracted chunks to English."""
        if not self.pdf_path:
            raise ValueError("No PDF path set")

        if self.prep_data and self.prep_data.get('translated'):
            return self.prep_data

        if not self.prep_data:
            cached = self._load_prep()
            if cached and cached.get('translated'):
                return cached

        print(f"\n{'='*80}")
        print(f"STEP 2: TRANSLATING")
        print(f"{'='*80}\n")

        extracted = self.extract_pdf()
        lang = extracted['language']
        chunks = extracted['chunks']

        if lang == 'en':
            print(f"✓ Already in English")
            self.prep_data['chunk_pairs'] = [
                {"original": c, "translated": c} for c in chunks]
            self.prep_data['translated_at'] = str(datetime.now())
        elif lang in self.HELSINKI_MODELS:
            start = time.time()
            translated_chunks = self._translate_chunks_helsinki(chunks, lang)
            elapsed = time.time() - start
            chunks_per_sec = len(translated_chunks) / elapsed
            print(
                f"✓ Translated {len(translated_chunks)} chunks in {elapsed:.1f}s ({chunks_per_sec:.2f} chunks/sec)")

            self.prep_data['chunks'] = translated_chunks
            self.prep_data['chunk_pairs'] = [
                {"original": o, "translated": t} for o, t in zip(chunks, translated_chunks)
            ]
            self.prep_data['translated'] = True
            self.prep_data['translation_model'] = 'helsinki'
            self.prep_data['translated_at'] = str(datetime.now())
            self._unload_translator()
        else:
            print(f"⚠ Translation not supported for: {lang}")
            self.prep_data['chunk_pairs'] = [
                {"original": c, "translated": c} for c in chunks]
            self.prep_data['unsupported'] = True
            self.prep_data['translated_at'] = str(datetime.now())

        self._save_prep()
        return self.prep_data

    # ==================== BERT ANALYSIS (UNCHANGED) ====================

    def load_model(self, model_name: str, task_name: str):
        """Load a BERT model for analysis."""
        if task_name in self.models:
            return self.models[task_name]

        if self.models:
            self._clear_gpu_memory()

        print(f"Loading {task_name} model: {model_name}")
        self.models[task_name] = pipeline(
            "text-classification",
            model=model_name,
            device=self.device,
            batch_size=self.BATCH_SIZE
        )
        return self.models[task_name]

    def filter_climate_chunks(self) -> Dict:
        """Filter for climate-related chunks using ClimateBERT."""
        if self.bert_data and self.bert_data.get('filtered'):
            return self.bert_data

        cached = self._load_bert()
        if cached and cached.get('filtered'):
            return cached

        if not self.prep_data:
            self.prep_data = self._load_prep()
            if not self.prep_data:
                raise FileNotFoundError("Run extract_pdf() first")

        if self.prep_data.get('translated'):
            chunks = self.prep_data['chunks']
            source = "translated"
        else:
            chunks = self.prep_data['chunks']
            source = "original"

        chunk_ids = self.prep_data.get(
            'chunk_ids', [f"chunk_{i:03d}" for i in range(len(chunks))])

        print("\nPreparing for BERT analysis...")
        self._unload_translator()
        self._clear_gpu_memory()

        if self.device >= 0:
            mem_allocated = torch.cuda.memory_allocated(0) / 1e9
            print(f"  GPU memory: {mem_allocated:.2f}GB allocated")

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector", "detector")

        print(
            f"\nFiltering {len(chunks)} {source} chunks for climate content...")
        climate_chunks = []

        for i in tqdm(range(0, len(chunks), self.BATCH_SIZE), desc="Detecting"):
            batch = chunks[i:i+self.BATCH_SIZE]
            batch_ids = chunk_ids[i:i+self.BATCH_SIZE]
            try:
                results = detector(batch, truncation=True, max_length=512)
                for chunk, chunk_id, result in zip(batch, batch_ids, results):
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'chunk_id': chunk_id,
                            'company': self.prep_data.get('company'),
                            'company_id': self.prep_data.get('company_id'),
                            'year': self.prep_data.get('year'),
                            'text': chunk,
                            'detector_score': result['score'],
                            'detector_label': result['label']
                        })
            except Exception as e:
                print(f"⚠ Error: {e}")
                self._clear_gpu_memory()

        kept_pct = len(climate_chunks)/len(chunks)*100 if chunks else 0
        print(f"✓ Kept {len(climate_chunks)} climate chunks ({kept_pct:.1f}%)")

        self.bert_data = {
            'pdf_path': str(self.pdf_path),
            'company': self.prep_data.get('company'),
            'company_id': self.prep_data.get('company_id'),
            'year': self.prep_data.get('year'),
            'language': self.prep_data.get('language', 'unknown'),
            'source': source,
            'filtered': True,
            'filtered_at': datetime.now().isoformat(),
            'total_chunks': len(chunks),
            'climate_chunks': len(climate_chunks),
            'kept_percentage': kept_pct,
            'chunks': climate_chunks
        }

        self._save_bert()
        return self.bert_data

    def analyze_specificity(self) -> Dict:
        """Analyze specificity of climate chunks."""
        if self.bert_data and self.bert_data.get('specificity_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data:
            self.filter_climate_chunks()

        chunks = self.bert_data['chunks']

        if 'detector' in self.models:
            del self.models['detector']
            self._clear_gpu_memory()

        specificity = self.load_model(
            "climatebert/distilroberta-base-climate-specificity", "specificity")

        print(f"\nAnalyzing specificity for {len(chunks)} chunks...")
        for i in tqdm(range(0, len(chunks), self.BATCH_SIZE), desc="Specificity"):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = specificity(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['specificity_label'] = result['label']
                    chunk['specificity_score'] = result['score']
            except Exception as e:
                print(f"⚠ Error: {e}")
                self._clear_gpu_memory()

        self.bert_data['specificity_analyzed'] = True
        self.bert_data['specificity_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def analyze_sentiment(self) -> Dict:
        """Analyze sentiment of climate chunks."""
        if self.bert_data and self.bert_data.get('sentiment_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data or not self.bert_data.get('specificity_analyzed'):
            self.analyze_specificity()

        chunks = self.bert_data['chunks']

        if 'specificity' in self.models:
            del self.models['specificity']
            self._clear_gpu_memory()

        sentiment = self.load_model(
            "climatebert/distilroberta-base-climate-sentiment", "sentiment")

        print(f"\nAnalyzing sentiment for {len(chunks)} chunks...")
        for i in tqdm(range(0, len(chunks), self.BATCH_SIZE), desc="Sentiment"):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = sentiment(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['sentiment_label'] = result['label']
                    chunk['sentiment_score'] = result['score']
            except Exception as e:
                print(f"⚠ Error: {e}")
                self._clear_gpu_memory()

        self.bert_data['sentiment_analyzed'] = True
        self.bert_data['sentiment_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def analyze_commitments(self) -> Dict:
        """Analyze commitments in climate chunks."""
        if self.bert_data and self.bert_data.get('commitment_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data or not self.bert_data.get('sentiment_analyzed'):
            self.analyze_sentiment()

        chunks = self.bert_data['chunks']

        if 'sentiment' in self.models:
            del self.models['sentiment']
            self._clear_gpu_memory()

        commitment = self.load_model(
            "climatebert/distilroberta-base-climate-commitment", "commitment")

        print(f"\nAnalyzing commitments for {len(chunks)} chunks...")
        for i in tqdm(range(0, len(chunks), self.BATCH_SIZE), desc="Commitments"):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = commitment(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['commitment_label'] = result['label']
                    chunk['commitment_score'] = result['score']
            except Exception as e:
                print(f"⚠ Error: {e}")
                self._clear_gpu_memory()

        self.bert_data['commitment_analyzed'] = True
        self.bert_data['commitment_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def process_pdfs(self, path: str, extract: bool = True, translate: bool = True,
                     filter_climate: bool = True, analyze: bool = True,
                     skip_errors: bool = True) -> Dict:
        """Process single PDF or batch of PDFs."""
        target = Path(path)
        if not target.exists():
            print(f"❌ Path not found: {path}")
            return None

        if target.is_file():
            if target.suffix.lower() != '.pdf':
                print(f"❌ Not a PDF file: {path}")
                return None
            pdf_files = [target]
            is_single = True
            root = target.parent
        else:
            pdf_files = sorted(target.rglob("*.pdf"))
            if not pdf_files:
                print(f"❌ No PDFs found in {path}")
                return None
            is_single = False
            root = target

        print(f"\n{'='*80}")
        print(f"{'PROCESSING' if is_single else 'BATCH PROCESSING'}")
        print(f"Found {len(pdf_files)} PDF(s)")
        print(f"Translation: Helsinki-NLP (fast)")
        print(f"{'='*80}\n")

        stats = {
            'total': len(pdf_files), 'extracted': 0, 'translated': 0,
            'skipped_translation': 0, 'filtered': 0, 'analyzed': 0, 'errors': [],
            'start_time': time.time()
        }

        for i, pdf_file in enumerate(pdf_files, 1):
            relative_path = pdf_file.relative_to(root)
            print(f"\n[{i}/{len(pdf_files)}] {relative_path}")

            try:
                self.set_pdf_path(str(pdf_file))

                if extract:
                    self.extract_pdf()
                    stats['extracted'] += 1

                if translate:
                    result = self.translate_pdf()
                    if result.get('translated'):
                        stats['translated'] += 1
                    else:
                        stats['skipped_translation'] += 1

                if filter_climate:
                    filtered = self.filter_climate_chunks()
                    stats['filtered'] += 1
                    print(
                        f"  ✓ {filtered['climate_chunks']}/{filtered['total_chunks']} climate chunks")

                if analyze and filter_climate:
                    self.analyze_specificity()
                    self.analyze_sentiment()
                    self.analyze_commitments()
                    stats['analyzed'] += 1

                self._clear_gpu_memory()

            except Exception as e:
                stats['errors'].append(
                    {'file': str(relative_path), 'error': str(e)})
                if not skip_errors:
                    raise
                print(f"  ⚠ Error: {e}")
                self._emergency_gpu_cleanup()

        stats['elapsed_time'] = time.time() - stats['start_time']
        stats['avg_time_per_pdf'] = stats['elapsed_time'] / \
            len(pdf_files) if pdf_files else 0

        print(f"\n{'='*80}")
        print(f"COMPLETE")
        print(f"  Extracted: {stats['extracted']}")
        print(f"  Translated: {stats['translated']}")
        print(f"  Filtered: {stats['filtered']}")
        print(f"  Analyzed: {stats['analyzed']}")
        print(f"  Errors: {len(stats['errors'])}")
        print(f"  Total time: {stats['elapsed_time']/60:.1f} min")
        print(f"  Avg per PDF: {stats['avg_time_per_pdf']:.1f}s")
        print(f"{'='*80}\n")

        return stats

    def inspect_extraction(self, n_samples=3):
        """Preview extracted chunks."""
        if not self.prep_data:
            self.prep_data = self._load_prep()
        if not self.prep_data:
            print("No data. Run extract_pdf() first.")
            return

        chunks = self.prep_data['chunks']
        lengths = [len(c) for c in chunks]

        print(f"\n{'='*80}")
        print(
            f"EXTRACTION: {self.pdf_path.name if self.pdf_path else 'unknown'}")
        print(f"Company: {self.prep_data.get('company')}")
        print(f"Year: {self.prep_data.get('year')}")
        print(f"Language: {self.prep_data['language']}")
        print(f"Method: {self.prep_data.get('extraction_method', 'unknown')}")
        print(f"Total chunks: {len(chunks)}")
        print(f"Avg size: {sum(lengths)/len(lengths):.0f} chars")
        print(f"Min/Max: {min(lengths)} / {max(lengths)} chars")
        print(f"{'='*80}\n")

        import random
        samples = random.sample(chunks, min(n_samples, len(chunks)))
        for i, chunk in enumerate(samples, 1):
            print(f"{'─'*80}")
            print(f"Sample {i} ({len(chunk)} chars):")
            print(f"{'─'*80}")
            print(chunk[:500] + ('...' if len(chunk) > 500 else ''))
            print()

    def get_final_results(self) -> Optional[Dict]:
        """Get final BERT analysis results."""
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data:
            print(f"❌ No analysis found")
            return None

        required = ['filtered', 'specificity_analyzed',
                    'sentiment_analyzed', 'commitment_analyzed']
        missing = [r for r in required if not self.bert_data.get(r)]
        if missing:
            print(f"⚠ Incomplete analysis. Missing: {', '.join(missing)}")

        return self.bert_data

## single file - step by step

In [30]:

analyzer = ClimateReportAnalyzer()

# pdf_path = "data/reports/SSAB/005_2015_annual_report.pdf"
# pdf_path = "data/reports/Celsa/007_2020_sustainability_report.pdf"
pdf_path = "data/reports/Dillinger/012_2019_annual_report.pdf"

# pdf_path = "data/reports/TataSteelNederland/006_2021_sustainability_report.pdf"
# pdf_path = "data/reports/Dillinger/012_2020_annual_report.pdf"

analyzer.process_pdfs(pdf_path)

analyzer.extract_pdf()
analyzer.translate_pdf()
analyzer.filter_climate_chunks()
analyzer.analyze_specificity()
analyzer.analyze_sentiment()
analyzer.analyze_commitments()
results = analyzer.get_final_results()

✓ GPU: NVIDIA T1200 Laptop GPU (3.9GB)
✓ PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True,garbage_collection_threshold:0.7,max_split_size_mb:256
✓ Translation backend: 'Helsinki-NLP (fast)'

PROCESSING
Found 1 PDF(s)
Translation: Helsinki-NLP (fast)


[1/1] 012_2019_annual_report.pdf

STEP 1: EXTRACTING PDF

✓ Metadata: company=Dillinger, id=012, year=2019


Extracting (PyMuPDF):   0%|          | 0/50 [00:00<?, ?it/s]

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
✓ Raw extraction: 123,632 characters
✓ After cleaning: 115,703 characters
✓ Language detected: de
✓ Created 64 chunks
  Chunk sizes: min=645, max=4295, avg=1803
✓ Cached prep: 012_2019_annual_report_prep.json

STEP 2: TRANSLATING

Loading Helsinki-NLP model: Helsinki-NLP/opus-mt-de-en
  Applying torch.compile()...
✓ Helsinki model loaded in 1.6s (0.16GB)
Translating de → en (Helsinki-NLP)
  Repetition mitigation: no_repeat_ngram_size=3, repetition_penalty=1.2, num_beams=4


Translating:   0%|          | 0/4 [00:00<?, ?it/s]

Post-processing translations...
✓ Translated 64 chunks in 98.5s (0.65 chunks/sec)
  Unloading translation model...
  GPU memory after unload: 0.01GB
✓ Cached prep: 012_2019_annual_report_prep.json

Preparing for BERT analysis...
  GPU memory: 0.01GB allocated
Loading detector model: climatebert/distilroberta-base-climate-detector


...

✓ Cached BERT analysis: 012_2019_annual_report_bert.json
Loading commitment model: climatebert/distilroberta-base-climate-commitment


Device set to use cuda:0



Analyzing commitments for 16 chunks...


Commitments:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Cached BERT analysis: 012_2019_annual_report_bert.json

COMPLETE
  Extracted: 1
  Translated: 1
  Filtered: 1
  Analyzed: 1
  Errors: 0
  Total time: 1.9 min
  Avg per PDF: 114.0s



## dir - whole pipe

In [21]:
%%time

# pdf_path = "data/reports/Celsa"
# pdf_path = "data/reports/LibertySteel"
pdf_path="data/reports/"
# pdf_path = "data/reports"

analyzer = ClimateReportAnalyzer()
stats = analyzer.process_pdfs(pdf_path)


✓ GPU: NVIDIA T1200 Laptop GPU (3.9GB)
✓ PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True,garbage_collection_threshold:0.7,max_split_size_mb:256
✓ Translation backend: Helsinki-NLP (fast)

BATCH PROCESSING
Found 204 PDF(s)
Translation: Helsinki-NLP (fast)


[1/204] AcciaieriedItalia/009_2021_sustainability_report.pdf

STEP 1: EXTRACTING PDF

✓ Metadata: company=AcciaieriedItalia, id=009, year=2021


Extracting (PyMuPDF):   0%|          | 0/107 [00:00<?, ?it/s]

✓ Raw extraction: 337,790 characters
✓ After cleaning: 289,422 characters
✓ Language detected: en
✓ Created 357 chunks
  Chunk sizes: min=600, max=1517, avg=791
✓ Cached prep: 009_2021_sustainability_report_prep.json

STEP 2: TRANSLATING

✓ Already in English
✓ Cached prep: 009_2021_sustainability_report_prep.json

Preparing for BERT analysis...
  GPU memory: 0.01GB allocated
Loading detector model: climatebert/distilroberta-base-climate-detector


Device set to use cuda:0



Filtering 357 original chunks for climate content...


...


Analyzing sentiment for 230 chunks...


Sentiment:   0%|          | 0/8 [00:00<?, ?it/s]

✓ Cached BERT analysis: 008_2024_corporate_responsibility_report_bert.json

Analyzing commitments for 230 chunks...


Commitments:   0%|          | 0/8 [00:00<?, ?it/s]

✓ Cached BERT analysis: 008_2024_corporate_responsibility_report_bert.json

COMPLETE
  Extracted: 204
  Translated: 23
  Filtered: 204
  Analyzed: 204
  Errors: 0
  Total time: 113.8 min
  Avg per PDF: 33.5s

CPU times: user 1h 50min 56s, sys: 5min 14s, total: 1h 56min 10s
Wall time: 1h 53min 45s
